# Gauteng 2026 LGE — Bayesian spatial VD/ward projection

Runs the full pipeline from [Zentannen/SAelection](https://github.com/Zentannen/SAelection) on Kaggle (CPU is enough; **Internet must be ON** in the notebook settings).

1. fetch IEC VD-level results (2016, 2019, 2021, 2024) and 2021/2024/2026 VD + ward boundaries
2. build contiguity (touching-VD) graphs
3. train the model with variational inference on past Gauteng elections (incl. hold-out test of the spatial term)
4. count-replay backtest on 2024
5. live projection of 2026 (upload IEC per-VD 2026 result CSVs as a Kaggle dataset to condition on the count)

In [ ]:
!git clone --depth 1 https://github.com/Zentannen/SAelection.git /kaggle/working/SAelection
%cd /kaggle/working/SAelection
!pip install -q -r requirements.txt

In [ ]:
!bash scripts/00_fetch_data.sh
!python scripts/prepare_data.py

In [ ]:
# Optional: condition on the live count. Attach a Kaggle dataset containing IEC 2026 per-VD CSVs
# (results.elections.org.za download format) and copy them in:
import glob, shutil, os
for f in glob.glob('/kaggle/input/*/*.csv'):
    shutil.copy(f, 'data/observed_2026/')
print(os.listdir('data/observed_2026'))

In [ ]:
# VI by default. For full HMC/NUTS instead (slow; ~35k latent dims) uncomment:
# import os; os.environ['SAELECTION_METHOD'] = 'nuts'; os.environ['SAELECTION_WARMUP'] = '500'
!python scripts/run_train.py


In [ ]:
!PYTHONPATH=src python -m saelection.backtest


In [ ]:
!python scripts/run_live.py

In [ ]:
import json, pandas as pd
print(json.dumps(json.load(open('output/live/meta.json'))['model']['spatial_term_kept'], indent=1))
pd.read_csv('output/live/municipalities.csv').head(10)

In [ ]:
# keep outputs in /kaggle/working so they appear under the notebook's Output tab
!cp -r output artifacts /kaggle/working/